<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/02-rdd-fundamentals/04-cache-persist-storage-levels.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 2.4 — cache(), persist(), storage levels

Watch the **Storage** tab of the UI fill and empty as you run this.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark import StorageLevel
import time

spark = (
    SparkSession.builder
    .appName("2.4-caching")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

## A deliberately expensive lineage

In [ ]:
def expensive_parse(x):
    for _ in range(400):            # simulate real parsing cost
        h = (x * 31 + 7) % 999983
    return (x % 100, h)

parsed = sc.parallelize(range(2_000_000), 8).map(expensive_parse)

def timed(label, f):
    t0 = time.perf_counter(); r = f(); dt = time.perf_counter() - t0
    print(f"{label:<28} {dt:6.2f}s")
    return r

In [ ]:
# Without caching: every action pays the full parse
timed("count, uncached #1", parsed.count)
timed("count, uncached #2", parsed.count)   # same price again

In [ ]:
parsed.cache()                               # lazy — Storage tab still empty
timed("count, materializes cache", parsed.count)   # pays parse + stores
timed("count, from cache", parsed.count)           # cheap
timed("distinct keys, from cache", parsed.keys().distinct().count)
# -> now check the Storage tab: the RDD, its size, Fraction Cached

## Cleanup matters

In [ ]:
parsed.unpersist()   # Storage tab empties
timed("count after unpersist", parsed.count)   # back to full price

## MEMORY_ONLY vs MEMORY_AND_DISK when data doesn't fit

We shrink usable memory indirectly by caching something bigger than the
storage pool. If Fraction Cached < 100% under MEMORY_ONLY, the missing
partitions are silently recomputed on every use; MEMORY_AND_DISK spills
them to local disk instead. (Whether it overflows on YOUR machine depends
on driver memory — see the lesson; bump the multiplier if needed.)

In [ ]:
big = sc.parallelize(range(30_000_000), 8).map(lambda x: (x, str(x) * 8))

big.persist(StorageLevel.MEMORY_ONLY)
timed("materialize MEMORY_ONLY", big.count)
timed("reuse      MEMORY_ONLY", big.count)
print("-> Storage tab: what's Fraction Cached?")
big.unpersist()

In [ ]:
big.persist(StorageLevel.MEMORY_AND_DISK)
timed("materialize MEM_AND_DISK", big.count)
timed("reuse      MEM_AND_DISK", big.count)
print("-> Storage tab: memory vs disk split for this RDD")
big.unpersist()

## Exercises

1. Re-run the `parsed` timing block but insert `parsed.take(3)` as the first action after `cache()`. Is the cache fully materialized afterwards (Storage tab)? Explain what you see.
2. Show that caching a once-used RDD is a net loss: time `parsed.count()` cold vs `parsed.cache(); parsed.count()` cold. Where does the extra time go?
3. Correctness bomb: build `r = sc.parallelize(range(10)).map(lambda x: (x, random.random()))`, cache it, `collect()` twice and compare. Then `unpersist()` and `collect()` again — did values change? Why?

In [ ]:
# your exercise code here
